<a href="https://colab.research.google.com/github/felixyustian/enterprise_ai_context_engine_fraud_risk_marketing/blob/main/enterprise_ai_context_engine_fraud_risk_marketing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# [CELL 1] Instalasi MCP SDK & Enterprise AI Stack
!pip install -qU mcp langchain langchain-google-genai langchain-community langgraph faiss-cpu pandas streamlit
!npm install -q -g localtunnel

print("✅ Infrastruktur MCP & AI Engine berhasil disiapkan!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 105.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently tak

In [2]:
%%writefile engine.py
import json
import pandas as pd
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END

# --- MCP SERVER: External Credit Bureau (Simulasi SLIK OJK/Pefindo) ---
class CreditBureauMCP:
    def __init__(self):
        # Database simulasi riwayat kredit calon debitur
        self.credit_db = pd.DataFrame({
            "app_id": ["APP-501", "APP-502", "APP-503"],
            "applicant_name": ["Budi Santoso", "Siti Aminah", "PT. Teknologi Bangsa"],
            "credit_score": [720, 580, 810],
            "debt_to_income_ratio": [0.25, 0.65, 0.15],
            "late_payments_24m": [0, 4, 0],
            "requested_amount_idr": [150000000, 50000000, 2000000000]
        })

    def call_tool(self, tool_name: str, arguments: dict):
        if tool_name == "get_credit_profile":
            app_id = arguments.get("app_id", "")
            data = self.credit_db[self.credit_db['app_id'] == app_id].to_dict('records')
            return json.dumps(data[0] if data else {"error": "Applicant Not Found in Bureau"})
        return json.dumps({"error": "Unknown tool"})

# --- AGENTIC CORE: Credit Underwriter ---
class UnderwritingState(TypedDict):
    app_id: str
    credit_data: str
    final_decision: str

def init_underwriting_engine(api_key: str):
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.1,
        api_key=api_key
    )

    mcp_server = CreditBureauMCP()

    # Node 1: Credit Data Fetcher (MCP Client)
    def fetch_credit_node(state: UnderwritingState) -> dict:
        print(f"🔗 [MCP] Menarik profil kredit untuk aplikasi {state['app_id']}...")
        raw_data = mcp_server.call_tool("get_credit_profile", {"app_id": state["app_id"]})
        return {"credit_data": raw_data}

    # Node 2: AI Underwriting Analyst
    def underwriting_node(state: UnderwritingState) -> dict:
        prompt = f"""
        Anda adalah AI Chief Credit Underwriter di sebuah Bank.

        Data Profil Kredit via MCP SLIK OJK: {state['credit_data']}

        Instruksi Evaluasi:
        1. Analisis kelayakan kredit berdasarkan 'credit_score' (Aman > 700), 'debt_to_income_ratio' (Maks 0.40), dan 'late_payments_24m' (Maks 1).
        2. Berikan Keputusan Akhir dalam format tag: [APPROVED], [REJECTED], atau [MANUAL_REVIEW].
        3. Susun alasan persetujuan/penolakan dalam poin-poin yang profesional dan berikan syarat tambahan jika berstatus MANUAL_REVIEW.
        """
        response = llm.invoke(prompt)
        return {"final_decision": response.content}

    # Merakit Graph
    workflow = StateGraph(UnderwritingState)
    workflow.add_node("Data_Fetcher", fetch_credit_node)
    workflow.add_node("Underwriter", underwriting_node)

    workflow.add_edge(START, "Data_Fetcher")
    workflow.add_edge("Data_Fetcher", "Underwriter")
    workflow.add_edge("Underwriter", END)

    return workflow.compile()

def run_credit_analysis(app_id: str, api_key: str):
    app = init_underwriting_engine(api_key)
    return app.invoke({"app_id": app_id})

Writing engine.py


In [3]:
%%writefile app.py
import streamlit as st
import engine
import json

st.set_page_config(page_title="AI Credit Underwriting", page_icon="🏦", layout="wide")

st.title("🏦 Enterprise AI Credit Underwriting Engine")
st.markdown("Skenario Bisnis: **Automated Loan Approval & Risk Assessment** | Architecture: **LangGraph + MCP**")
st.divider()

with st.sidebar:
    st.header("⚙️ Konfigurasi Engine")
    api_key = st.text_input("Gemini API Key", type="password")

    st.header("📡 Credit Bureau Logs (MCP)")
    json_log = st.empty()

    st.info("💡 **Tips Data Mock:** \n- `APP-501` (Kredit Baik)\n- `APP-502` (Kredit Macet)\n- `APP-503` (Korporasi)")

col1, col2 = st.columns([1, 1.5])

with col1:
    st.subheader("Sistem Pengajuan")
    app_input = st.text_input("Masukkan ID Aplikasi Kredit:", value="APP-502")

    if st.button("⚖️ Evaluasi Kelayakan Kredit", use_container_width=True):
        if not api_key:
            st.error("⚠️ API Key wajib diisi untuk menjalankan AI Underwriter.")
        else:
            with st.spinner("Memanggil data SLIK OJK/Biro Kredit via MCP..."):
                try:
                    result = engine.run_credit_analysis(app_input, api_key)

                    # Log Visualisasi Data
                    credit_data_parsed = json.loads(result['credit_data'])
                    json_log.json({
                        "mcp_protocol": "v1.0",
                        "endpoint": "get_credit_profile",
                        "response": credit_data_parsed
                    })

                    with col2:
                        st.subheader("📑 Dokumen Keputusan Kredit")
                        if "error" in credit_data_parsed:
                            st.error("❌ Data pemohon tidak ditemukan di Biro Kredit.")
                        else:
                            st.success("Analisis Underwriting Selesai")
                            st.markdown(result['final_decision'])
                except Exception as e:
                    st.error(f"Terjadi kesalahan sistem: {e}")

Writing app.py


In [4]:
# [CELL 4] Menjalankan Server UI via Google Proxy
import time
import subprocess
from google.colab import output

print("🚀 Memulai AI Underwriting Engine...")
!pkill -f streamlit
time.sleep(2)

subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true"
])
time.sleep(4)

print("🛡️ Membangun jalur aman via infrastruktur internal Google...")
output.serve_kernel_port_as_window(8501)

🚀 Memulai AI Underwriting Engine...
🛡️ Membangun jalur aman via infrastruktur internal Google...
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>